# Notebook 10b — Phase Composite Precipitation Maps
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

Two composite diagnostics comparing three 2-D representations.

## Part A — Basic phase composites
For each phase label (1–8): collect all days assigned that phase → average Lee-preprocessed
precipitation anomaly → one map.  
**Answers:** *"When BSISO is in phase X, where is it wet/dry?"*

## Part B — ENSO-stratified (EN − LN) difference composites
Within each phase group, split by ENSO category (El Niño / La Niña / Neutral).  
EN mean − LN mean → 8 difference maps per representation.  
**Answers:** *"How does ENSO modulate the BSISO precipitation pattern at each phase?"*

## Phase label sources
| Repr | Phase labels | Rationale |
|------|-------------|----------|
| `idx` | `bsiso_phase` column (APEC BSISO 1–8) | idx IS the BSISO index |
| `sup` | `bsiso_phase` column (same CSV) | sup trained on these labels; ρ_c=0.844 |
| `ssl` | θ_ssl binned into 8 equal 45° sectors | SSL reverses direction; must use own geometry |

**Why ssl cannot use BSISO labels:** compositing ssl-days by BSISO phase just reproduces the idx/sup
composite on a smaller dataset. SSL's representation would play no role.
Since ρ_c(idx,ssl)=−0.305, SSL sector k maps approximately to BSISO phase (9−k) mod 8
(reversed order, but spatially coherent).

**Outputs:**
```
results/precip_composite/
  phase_composites.png       — 3×8 basic composite maps
  enso_diff_composites.png   — 3×8 EN−LN difference maps
  sample_counts.csv          — N per (repr, phase, enso) cell
  composite_report.txt       — plain-text summary
```

## Cell 1 — Mount Drive & Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
RAW_DIR       = f'{PROJECT_DIR}/data/raw'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
RESULTS_DIR   = f'{PROJECT_DIR}/results'
OUT_DIR       = f'{RESULTS_DIR}/precip_composite'
os.makedirs(OUT_DIR, exist_ok=True)

SUP_EMB_FILE    = f'{RESULTS_DIR}/lee_2d_no_l2/embeddings.npy'
SSL_EMB_FILE    = f'{RESULTS_DIR}/lee_2d_ssl/embeddings.npy'
LABELS_SUP_FILE = f'{PROCESSED_DIR}/labels_aligned_mjjas_lee.csv'
LABELS_SSL_FILE = f'{PROCESSED_DIR}/labels_aligned_mjjas_lee_lp25.csv'
BSISO_RAW_FILE  = f'{RAW_DIR}/BSISO.INDEX.NORM.LY.data'
PRECIP_FILE     = f'{RAW_DIR}/precip_MJJAS_1979_2023.nc'

for f in [SUP_EMB_FILE, SSL_EMB_FILE, LABELS_SUP_FILE, LABELS_SSL_FILE,
          BSISO_RAW_FILE, PRECIP_FILE]:
    print(f'[{"OK" if os.path.exists(f) else "MISSING"}] {os.path.basename(f)}')

## Cell 2 — Load Embeddings & Build Phase Labels

In [ ]:
# ── supervised: load embeddings + BSISO labels ────────────────────────────
emb_sup = np.load(SUP_EMB_FILE)          # (N_sup, 2)
df_sup  = pd.read_csv(LABELS_SUP_FILE, parse_dates=['date'])
df_sup['date'] = df_sup['date'].dt.normalize()
assert len(emb_sup) == len(df_sup)

# idx and sup share the same BSISO phase labels
dates_sup     = pd.DatetimeIndex(df_sup['date'])
phase_idx     = df_sup['bsiso_phase'].values.astype(int)   # 1–8
phase_sup     = phase_idx.copy()                            # same labels
enso_sup      = df_sup['enso_category'].values              # 'El Nino'/'La Nina'/'Neutral'
amplitude_sup = df_sup['bsiso_amplitude'].values.astype(float)

# Sanity check: print actual unique strings so any future mismatch is obvious
print('enso_category unique values (sup):', np.unique(enso_sup))

# ── SSL: load embeddings + ssl-day labels ────────────────────────────────
emb_ssl = np.load(SSL_EMB_FILE)          # (N_ssl, 2)
df_ssl  = pd.read_csv(LABELS_SSL_FILE, parse_dates=['date'])
df_ssl['date'] = df_ssl['date'].dt.normalize()
assert len(emb_ssl) == len(df_ssl)

dates_ssl = pd.DatetimeIndex(df_ssl['date'])
enso_ssl  = df_ssl['enso_category'].values
print('enso_category unique values (ssl):', np.unique(enso_ssl))

# SSL phase labels: θ_ssl binned into 8 equal 45° sectors (−π to π)
theta_ssl = np.arctan2(emb_ssl[:, 1], emb_ssl[:, 0])
# Bin edges: -π, -3π/4, -π/2, -π/4, 0, π/4, π/2, 3π/4, π
bin_edges  = np.linspace(-np.pi, np.pi, 9)
# np.digitize: returns 1–8 for values in [-π, π) (clip the π boundary)
theta_ssl_clipped = np.clip(theta_ssl, -np.pi, np.pi - 1e-9)
phase_ssl = np.digitize(theta_ssl_clipped, bin_edges[1:-1]) + 1  # 1-indexed, 1–8

# ── summary ───────────────────────────────────────────────────────────────
print(f'\nidx/sup  N={len(phase_idx)}, phases: {np.unique(phase_idx)}')
print(f'ssl      N={len(phase_ssl)}, sectors: {np.unique(phase_ssl)}')
print()
print('Sample counts per phase:')
print(f'{"Phase/Sector":<14}', '  '.join(f'{p:>5}' for p in range(1, 9)))
for name, phases in [('idx/sup', phase_idx), ('ssl', phase_ssl)]:
    counts = [int((phases == p).sum()) for p in range(1, 9)]
    print(f'{name:<14}', '  '.join(f'{c:>5}' for c in counts))

# Build date→row-index lookup for precipitation
print('\nSSL sector–BSISO phase correspondence (reversed rotation expected):')
for s in range(1, 9):
    mask = phase_ssl == s
    ssl_dates_in_sector = set(dates_ssl[mask])
    sup_mask = np.array([d in ssl_dates_in_sector for d in dates_sup])
    if sup_mask.sum() > 0:
        modal_phase = int(pd.Series(phase_idx[sup_mask]).mode()[0])
        print(f'  SSL sector {s}  →  BSISO phase modal={modal_phase}  '
              f'(N={mask.sum()})')
    else:
        print(f'  SSL sector {s}  →  no overlap with sup dates (N={mask.sum()})')

## Cell 3 — Load & Lee-Preprocess Precipitation

In [ ]:
import xarray as xr

ds = xr.open_dataset(PRECIP_FILE)
time_dim = 'valid_time' if 'valid_time' in ds.dims else 'time'
tp_var   = 'tp' if 'tp' in ds.data_vars else list(ds.data_vars)[0]

times_raw = pd.DatetimeIndex(ds[time_dim].values).normalize()
tp_raw    = np.clip(ds[tp_var].values.astype(np.float32), 0, None)
lats      = ds.latitude.values
lons      = ds.longitude.values
ds.close()
T, nlat, nlon = tp_raw.shape

# ── Step 1: 3-harmonic Fourier annual cycle removal (clim 1981–2010) ──────
CLIM_START, CLIM_END = 1981, 2010
clim_mask  = np.isin(times_raw.year, range(CLIM_START, CLIM_END + 1))
times_clim = times_raw[clim_mask]
tp_clim    = tp_raw[clim_mask]

doys     = np.arange(1, 367)
clim_map = np.zeros((len(doys), nlat, nlon), dtype=np.float32)
counts   = np.zeros(len(doys), dtype=int)
for t_idx, d in enumerate(times_clim.dayofyear):
    clim_map[d - 1] += tp_clim[t_idx]
    counts[d - 1]   += 1
for d in range(len(doys)):
    if counts[d] > 0:
        clim_map[d] /= counts[d]

t_rad  = 2 * np.pi * doys / 365.25
basis  = np.column_stack([np.ones(len(doys))] +
                         [f(k * t_rad) for k in range(1, 4)
                          for f in (np.cos, np.sin)])
coef, *_ = np.linalg.lstsq(basis, clim_map.reshape(len(doys), -1), rcond=None)
clim_fit  = (basis @ coef).reshape(len(doys), nlat, nlon).astype(np.float32)

doy_all = times_raw.dayofyear
tp_anom = tp_raw - clim_fit[doy_all - 1]

# ── Step 2: subtract preceding 120-day running mean ───────────────────────
W = 120
tp_rm = np.zeros_like(tp_anom)
for i in range(1, T):
    tp_rm[i] = tp_anom[max(0, i - W):i].mean(axis=0)
tp_anom2 = tp_anom - tp_rm

# ── Step 3: normalize by area-averaged temporal std ───────────────────────
area_std = tp_anom2.std(axis=0).mean()
tp_norm  = (tp_anom2 / area_std).astype(np.float32)

# Build date lookup
date_to_tp_idx = {d: i for i, d in enumerate(times_raw)}

print(f'tp_norm: shape={tp_norm.shape}, area_std={area_std:.6f} m')
print('Precipitation preprocessing complete.')

## Cell 4 — Part A: Basic Phase Composites (3 repr × 8 phases)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

def compute_phase_composites(dates, phases, n_phases, date_to_tp_idx, tp_norm,
                              min_samples=10):
    """Return dict: label -> (mean_tp_map, count)."""
    nlat, nlon = tp_norm.shape[1], tp_norm.shape[2]
    composites = {}
    for p in range(1, n_phases + 1):
        mask    = phases == p
        tp_rows = [tp_norm[date_to_tp_idx[d]] for d in dates[mask]
                   if d in date_to_tp_idx]
        if len(tp_rows) >= min_samples:
            composites[p] = (np.stack(tp_rows).mean(axis=0), len(tp_rows))
        else:
            composites[p] = (np.full((nlat, nlon), np.nan), len(tp_rows))
    return composites


reprs_info = [
    ('BSISO Index (idx)',       dates_sup, phase_idx),
    ('Supervised 2D (sup)',     dates_sup, phase_sup),
    ('SSL 2D (ssl θ-sectors)', dates_ssl, phase_ssl),
]

all_composites = {}
for label, dates, phases in reprs_info:
    all_composites[label] = compute_phase_composites(
        dates, phases, 8, date_to_tp_idx, tp_norm)

# ── build SSL sector → modal BSISO phase mapping ──────────────────────────
# For each SSL sector, find which BSISO phase those days most commonly fall in.
# This is the same cross-check as Cell 2, but stored for reuse in plotting.
ssl_sector_to_bsiso = {}
for s in range(1, 9):
    mask_s  = phase_ssl == s
    s_dates = set(dates_ssl[mask_s])
    sup_mask = np.array([d in s_dates for d in dates_sup])
    if sup_mask.sum() > 0:
        ssl_sector_to_bsiso[s] = int(pd.Series(phase_idx[sup_mask]).mode()[0])
    else:
        ssl_sector_to_bsiso[s] = s  # fallback (shouldn't happen)

# Sort SSL sectors by their modal BSISO phase (1→8) so that
# column k of the SSL row corresponds to the same BSISO phase as
# column k of the idx/sup rows.
ssl_display_order = sorted(range(1, 9), key=lambda s: ssl_sector_to_bsiso[s])

print('SSL panel re-ordering (aligning with BSISO phases):')
print(f'  {"Column (=BSISO phase)":<25} {"SSL sector shown":<18} {"Modal BSISO phase of that sector"}')
for col, s in enumerate(ssl_display_order):
    print(f'  {col+1:<25} {s:<18} {ssl_sector_to_bsiso[s]}')

# ── plot ──────────────────────────────────────────────────────────────────
VMAX   = 0.3
cmap   = plt.cm.RdBu_r
norm   = mcolors.TwoSlopeNorm(vmin=-VMAX, vcenter=0, vmax=VMAX)
extent = [lons.min(), lons.max(), lats.min(), lats.max()]

fig, axes = plt.subplots(3, 8, figsize=(28, 10))
fig.suptitle(
    'Phase Composite Precipitation Maps\n'
    'Lee-preprocessed tp anomaly (normalized)  |  1979–2023 MJJAS\n'
    'SSL panels reordered to align with BSISO phase 1–8',
    fontsize=12, fontweight='bold')

row_labels = [
    'BSISO Index (idx)\nPhase 1–8',
    'Supervised 2D (sup)\nPhase 1–8 (same labels)',
    'SSL 2D (ssl)\nθ_ssl sectors\n(reordered → BSISO phase)',
]

for row, (label, dates, phases) in enumerate(reprs_info):
    comps = all_composites[label]
    for col in range(8):
        ax = axes[row, col]

        if row == 2:   # SSL: use reordered sectors
            sector      = ssl_display_order[col]
            bsiso_equiv = ssl_sector_to_bsiso[sector]
            comp_map, n = comps[sector]
            title_str   = f'Sec.{sector}→Ph.{bsiso_equiv}  (N={n})'
        else:
            p           = col + 1
            comp_map, n = comps[p]
            title_str   = f'Phase {p}  (N={n})'

        im = ax.imshow(comp_map, cmap=cmap, norm=norm,
                       origin='upper', extent=extent, aspect='auto')
        rect = Rectangle((100, 20), 45, 25, linewidth=1,
                         edgecolor='k', facecolor='none', linestyle='--')
        ax.add_patch(rect)
        ax.set_title(title_str, fontsize=7.5)
        ax.set_xticks([])
        ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(row_labels[row], fontsize=8)

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
plt.colorbar(im, cax=cbar_ax, label='Normalized tp anomaly')

out_basic = f'{OUT_DIR}/phase_composites.png'
plt.savefig(out_basic, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_basic}')

## Cell 5 — Part B: ENSO-Stratified (EN − LN) Difference Maps

In [ ]:
def compute_enso_diff(dates, phases, enso_cats, n_phases,
                      date_to_tp_idx, tp_norm, min_samples=5):
    """
    Returns:
      diff_maps : dict phase -> (nlat, nlon) EN_mean - LN_mean
      counts    : dict phase -> {'EN': n, 'LN': n}

    ENSO category strings in CSV are 'El Nino' / 'La Nina' / 'Neutral'
    (no tilde — matches notebook 02 classify_enso()).
    """
    nlat, nlon = tp_norm.shape[1], tp_norm.shape[2]
    diff_maps  = {}
    counts     = {}
    for p in range(1, n_phases + 1):
        phase_mask = phases == p
        en_rows, ln_rows = [], []
        for i, d in enumerate(dates):
            if not phase_mask[i]:
                continue
            if d not in date_to_tp_idx:
                continue
            cat = enso_cats[i]
            row = tp_norm[date_to_tp_idx[d]]
            if cat == 'El Nino':
                en_rows.append(row)
            elif cat == 'La Nina':
                ln_rows.append(row)
        n_en, n_ln = len(en_rows), len(ln_rows)
        counts[p] = {'EN': n_en, 'LN': n_ln}
        if n_en >= min_samples and n_ln >= min_samples:
            diff_maps[p] = np.stack(en_rows).mean(axis=0) - np.stack(ln_rows).mean(axis=0)
        else:
            diff_maps[p] = np.full((nlat, nlon), np.nan)
    return diff_maps, counts


enso_info = [
    ('BSISO Index (idx)',       dates_sup, phase_idx, enso_sup),
    ('Supervised 2D (sup)',     dates_sup, phase_sup, enso_sup),
    ('SSL 2D (ssl θ-sectors)', dates_ssl, phase_ssl, enso_ssl),
]

all_diff   = {}
all_counts = {}
for label, dates, phases, enso in enso_info:
    diff_maps, counts = compute_enso_diff(
        dates, phases, enso, 8, date_to_tp_idx, tp_norm)
    all_diff[label]   = diff_maps
    all_counts[label] = counts
    print(f'{label}:')
    for p in range(1, 9):
        c = counts[p]
        print(f'  sector/phase {p}: EN={c["EN"]:>3}  LN={c["LN"]:>3}')

# ── plot (SSL panels reordered using ssl_display_order from Cell 4) ───────
VMAX2 = 0.4
norm2 = mcolors.TwoSlopeNorm(vmin=-VMAX2, vcenter=0, vmax=VMAX2)

fig2, axes2 = plt.subplots(3, 8, figsize=(28, 10))
fig2.suptitle(
    'El Niño − La Niña Precipitation Difference by Phase\n'
    'Lee-preprocessed tp anomaly  |  1979–2023 MJJAS\n'
    'SSL panels reordered to align with BSISO phase 1–8',
    fontsize=12, fontweight='bold')

row_labels = [
    'BSISO Index (idx)\nPhase 1–8',
    'Supervised 2D (sup)\nPhase 1–8 (same labels)',
    'SSL 2D (ssl)\nθ_ssl sectors\n(reordered → BSISO phase)',
]

for row, (label, dates, phases, enso) in enumerate(enso_info):
    diff_maps = all_diff[label]
    counts    = all_counts[label]
    for col in range(8):
        ax = axes2[row, col]

        if row == 2:   # SSL: use reordered sectors
            sector      = ssl_display_order[col]
            bsiso_equiv = ssl_sector_to_bsiso[sector]
            dmap        = diff_maps[sector]
            c           = counts[sector]
            title_str   = f'Sec.{sector}→Ph.{bsiso_equiv}\nEN={c["EN"]} LN={c["LN"]}'
        else:
            p     = col + 1
            dmap  = diff_maps[p]
            c     = counts[p]
            title_str = f'Phase {p}\nEN={c["EN"]} LN={c["LN"]}'

        if np.isnan(dmap).all():
            title_str += '\n[insuf.]'

        im2 = axes2[row, col].imshow(dmap, cmap=cmap, norm=norm2,
                                     origin='upper', extent=extent, aspect='auto')
        rect = Rectangle((100, 20), 45, 25, linewidth=1,
                         edgecolor='k', facecolor='none', linestyle='--')
        ax.add_patch(rect)
        ax.set_title(title_str, fontsize=7.5)
        ax.set_xticks([])
        ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(row_labels[row], fontsize=8)

fig2.subplots_adjust(right=0.92)
cbar_ax2 = fig2.add_axes([0.93, 0.15, 0.015, 0.7])
plt.colorbar(im2, cax=cbar_ax2, label='EN − LN normalized tp anomaly')

out_enso = f'{OUT_DIR}/enso_diff_composites.png'
plt.savefig(out_enso, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_enso}')

## Cell 6 — Report & Save Sample Counts

In [ ]:
import datetime

# ── sample counts CSV ─────────────────────────────────────────────────────
count_rows = []
for label, dates, phases, enso in enso_info:
    for p in range(1, 9):
        c = all_counts[label][p]
        _, n_total = all_composites[label][p]
        count_rows.append({'repr': label.split('(')[1].rstrip(')'),
                            'phase_sector': p,
                            'N_total': n_total,
                            'N_EN': c['EN'],
                            'N_LN': c['LN']})
df_counts = pd.DataFrame(count_rows)
df_counts.to_csv(f'{OUT_DIR}/sample_counts.csv', index=False)
print(df_counts.to_string(index=False))

# ── plain-text report ─────────────────────────────────────────────────────
lines = [
    '=' * 65,
    'PLAN 3b — Phase Composite Precipitation Report',
    f'Date: {datetime.date.today()}',
    '=' * 65,
    '',
    'Part A — Basic phase composites',
    '  idx: BSISO phase labels 1–8  (N_total ~6579 total)',
    '  sup: same BSISO phase labels (identical days to idx)',
    '  ssl: θ_ssl sectors 1–8  (N_total ~4429 post-bandpass)',
    '',
    'Part B — EN − LN difference composites',
    '  ENSO category from enso_category column in CSV',
    '  min_samples = 5 per (phase, ENSO) cell to plot',
    '',
    'SSL sector → BSISO phase mapping (reversed rotation expected):',
]
for s in range(1, 9):
    bsiso_approx = ((9 - s) % 8) or 8
    lines.append(f'  SSL sector {s}  ≈  BSISO phase {bsiso_approx} (theoretical)')

lines += [
    '',
    'Output files:',
    f'  {OUT_DIR}/phase_composites.png',
    f'  {OUT_DIR}/enso_diff_composites.png',
    f'  {OUT_DIR}/sample_counts.csv',
    '=' * 65,
]
report_text = '\n'.join(lines)
print()
print(report_text)

with open(f'{OUT_DIR}/composite_report.txt', 'w') as fh:
    fh.write(report_text + '\n')
print('\nReport saved.')

---
## Done

**How to interpret the figures:**

**Part A (phase_composites.png):**  
Rows 1–2 (idx, sup) should look nearly identical (same BSISO phase labels, same days).  
Row 3 (ssl): composites should appear in roughly reversed phase order relative to rows 1–2
(SSL sector 1 ≈ BSISO phase 8, sector 2 ≈ phase 7, etc.).  
If both rows 1–2 and row 3 show spatially coherent alternating wet/dry patterns,  
both representations have discovered physically meaningful angular structures.

**Part B (enso_diff_composites.png):**  
Red = El Niño wetter than La Niña at that phase. Blue = El Niño drier.  
If SSL's EN−LN difference maps show larger amplitudes or cleaner spatial structure  
than idx/sup, this is direct evidence that the SSL ENSO signal (z=14.55) translates  
into real precipitation predictability — connecting Plan 1 results to Plan 3.

---
*DDCS Project | jh9141@nyu.edu*